# EODHD News - Sentiment Features

Builds `data/news/sentiment_features/{TICKER}.csv` for the 5 paper tickers.

Each file: `date` + `{company}_polarity` + `{company}_log_count` for target + 10 related companies

## 1. Setup

In [2]:
import os, ast, json, time
import numpy as np
import pandas as pd
import requests
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

load_dotenv(ROOT / '.env', override=True)
EODHD_API_TOKEN = (os.getenv('EODHD_API_TOKEN') or '').strip()
if not EODHD_API_TOKEN:
    raise RuntimeError('EODHD_API_TOKEN not set in .env')

DATA_DIR       = ROOT / 'data' / 'news'
CHECKPOINT_DIR = DATA_DIR / 'per_ticker_paper'
SENTIMENT_DIR  = DATA_DIR / 'sentiment_features'
SENTIMENT_DIR.mkdir(parents=True, exist_ok=True)

NEWS_FROM         = '2016-05-01'
NEWS_TO           = '2024-05-08'
REQUEST_SLEEP_SEC = 0.3

print('ROOT          :', ROOT)
print('Sentiment dir :', SENTIMENT_DIR)
print('Token OK      :', bool(EODHD_API_TOKEN))
print('Date range    :', NEWS_FROM, '->', NEWS_TO)

ROOT          : /Users/tmq/Documents/GitHub/stocks-prediction
Sentiment dir : /Users/tmq/Documents/GitHub/stocks-prediction/data/news/sentiment_features
Token OK      : True
Date range    : 2016-05-01 -> 2024-05-08


## 2. Fetch all 5 paper tickers

Uses EODHD `from`/`to` date filter to only articles in the paper's date range.  
Checkpoints saved to `data/news/per_ticker_paper/`. Delete a CSV to re-fetch.

**Symbol notes**:
- `AAPL/HSBC/PEP/TM` to `.US` (US-listed or ADR with clean coverage)
- `TCEHY` to fetched via `0700.HK` (Tencent primary HK listing) because `TCEHY.US` ADR has poor coverage. Saved under `ticker='TCEHY'` for consistency with price data.

In [3]:
PAPER_SYMBOLS = {
    'AAPL':  'AAPL.US',
    'HSBC':  'HSBC.US',
    'PEP':   'PEP.US',
    'TM':    'TM.US',
    'TCEHY': '0700.HK',
}

if False: 
    NEWS_FROM_TS = pd.Timestamp(NEWS_FROM, tz='UTC')
    NEWS_TO_TS   = pd.Timestamp(NEWS_TO,   tz='UTC')

    def fetch_ticker(ticker: str, force: bool = False) -> None:
        symbol = PAPER_SYMBOLS[ticker]
        out = CHECKPOINT_DIR / f'{ticker}.csv'
        if (not force) and out.exists():
            n = max(0, sum(1 for _ in open(out)) - 1)
            print(f'[{ticker}] checkpoint exists ({n:,} rows) -> skip')
            return
        print(f'[{ticker}] fetching {symbol} ({NEWS_FROM} -> {NEWS_TO}) ...')
        articles, offset = [], 0
        while True:
            r = requests.get('https://eodhd.com/api/news', params={
                's': symbol, 'limit': 100, 'offset': offset,
                'from': NEWS_FROM, 'to': NEWS_TO,
                'api_token': EODHD_API_TOKEN, 'fmt': 'json',
            }, timeout=60)
            if r.status_code != 200:
                print(f'  HTTP {r.status_code} at offset={offset}')
                break
            batch = r.json()
            if not isinstance(batch, list) or not batch:
                print(f'  done (total={len(articles):,} articles)')
                break
            articles.extend(batch)
            print(f'  offset={offset:>6}: +{len(batch)} -> total={len(articles):,}')
            if len(batch) < 100:
                break
            offset += 100
            time.sleep(REQUEST_SLEEP_SEC)
        df = pd.DataFrame(articles) if articles else pd.DataFrame()
        if not df.empty:
            df['ticker'] = ticker
        df.to_csv(out, index=False)
        print(f'Saved {out.name} ({len(df):,} rows)')

    for ticker in PAPER_SYMBOLS:
        fetch_ticker(ticker)

In [4]:
# Load raw CSVs from per_ticker_paper into memory
raw_dfs = {}
for ticker in ['AAPL', 'HSBC', 'PEP', 'TM', 'TCEHY']:
    df = pd.read_csv(CHECKPOINT_DIR / f'{ticker}.csv')
    raw_dfs[ticker] = df
    print(f'[{ticker}] {len(df):6,} rows | {df.columns.tolist()}')

[AAPL] 27,089 rows | ['date', 'title', 'content', 'link', 'symbols', 'tags', 'sentiment', 'ticker']
[HSBC]  2,017 rows | ['date', 'title', 'content', 'link', 'symbols', 'tags', 'sentiment', 'ticker']
[PEP]  3,693 rows | ['date', 'title', 'content', 'link', 'symbols', 'tags', 'sentiment', 'ticker']
[TM]  3,833 rows | ['date', 'title', 'content', 'link', 'symbols', 'tags', 'sentiment', 'ticker']
[TCEHY]  1,674 rows | ['date', 'title', 'content', 'link', 'symbols', 'tags', 'sentiment', 'ticker']


## 3. Parse target sentiment from per_ticker_paper

In [5]:
def _parse_polarity(s):
    if isinstance(s, dict):
        return s.get('polarity')
    if isinstance(s, str):
        try:
            d = ast.literal_eval(s)
            return d.get('polarity') if isinstance(d, dict) else None
        except Exception:
            return None
    return None

target_daily = {}
for ticker in ['AAPL', 'HSBC', 'PEP', 'TM', 'TCEHY']:
    raw = pd.read_csv(CHECKPOINT_DIR / f'{ticker}.csv')
    raw['date'] = pd.to_datetime(raw['date'], utc=True).dt.normalize().dt.tz_localize(None)
    raw['polarity'] = raw['sentiment'].apply(_parse_polarity)
    raw = raw.dropna(subset=['polarity'])
    daily = (raw.groupby('date', as_index=False)
               .agg(polarity=('polarity', 'mean'), count=('polarity', 'size')))
    target_daily[ticker] = daily
    print(f'[{ticker}] {len(daily):4d} days | {daily.date.min().date()} -? {daily.date.max().date()}')

[AAPL] 1374 days | 2017-10-05 -? 2024-05-08
[HSBC]  746 days | 2017-06-28 -? 2024-05-08
[PEP] 1013 days | 2017-10-24 -? 2024-05-08
[TM]  990 days | 2016-06-22 -? 2024-05-08
[TCEHY]  587 days | 2017-06-15 -? 2024-05-08


## 4. Fetch related company sentiment

In [6]:
RELATED = {
    'AAPL':  ['GOOGL.US', 'AMZN.US', 'MSFT.US', 'TSLA.US', 'SSNLF.US',
              'NVDA.US', 'META.US', 'INTC.US', 'AMD.US',  'IBM.US'],
    'HSBC':  ['JPM.US',   'SCBFF.US', 'MS.US',  'UBS.US',  'C.US',
              'GS.US',    'CS.US',   'NWG.US',  'AAPL.US', 'MSFT.US'],
    'PEP':   ['KO.US',    'WMT.US',  'COST.US', 'CL.US',   'CVX.US',
              'PG.US',    'JNJ.US',  'AAPL.US', 'MSFT.US', 'AMZN.US', 'TSLA.US', 'IBM.US'],
    'TM':    ['NSANY.US', 'HMC.US',  'MZDAY.US','F.US',    'GM.US',
              'VWAPY.US', 'BMWYY.US','TSLA.US', 'HYMTF.US','BYDDY.US', 'AAPL.US', 'AMZN.US'],
    'TCEHY': ['NTES.US',  'BIDU.US', 'JD.US',   'BABA.US', 'MSFT.US',
              'AAPL.US',  'GOOGL.US','AMZN.US', 'META.US', 'NVDA.US', 'INTC.US'],
}

RELATED_CACHE = DATA_DIR / 'related_sentiments_raw.json'
all_related   = sorted({s for syms in RELATED.values() for s in syms})
print(f'{len(all_related)} unique related tickers')

if RELATED_CACHE.exists():
    with open(RELATED_CACHE) as f:
        related_raw = json.load(f)
    print(f'Loaded from cache ({len(related_raw)} tickers)')
else:
    related_raw = {}
    for i in range(0, len(all_related), 10):
        batch = all_related[i:i+10]
        r = requests.get('https://eodhd.com/api/sentiments', params={
            's': ','.join(batch), 'from': NEWS_FROM, 'to': NEWS_TO,
            'api_token': EODHD_API_TOKEN, 'fmt': 'json',
        }, timeout=60)
        data = r.json()
        related_raw.update(data)
        n = sum(len(v) for v in data.values() if isinstance(v, list))
        print(f'  batch {i//10+1}: {n} rows')
        time.sleep(REQUEST_SLEEP_SEC)
    with open(RELATED_CACHE, 'w') as f:
        json.dump(related_raw, f)
    print(f'Saved -> {RELATED_CACHE.name}')

39 unique related tickers
Loaded from cache (39 tickers)


## 5. Build sentiment feature files

In [7]:
DATE_IDX = pd.date_range(NEWS_FROM, NEWS_TO, freq='D', name='date')


def _align_target(daily_df):
    s = daily_df.set_index('date')
    return (
        s['polarity'].reindex(DATE_IDX, fill_value=0.0),
        s['count'].reindex(DATE_IDX, fill_value=0.0),
    )


def _align_related(records):
    if not records:
        z = pd.Series(0.0, index=DATE_IDX)
        return z, z.copy()
    df = pd.DataFrame(records)
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date')
    return (
        df['normalized'].reindex(DATE_IDX, fill_value=0.0),
        df['count'].reindex(DATE_IDX, fill_value=0.0),
    )


for ticker, related_syms in RELATED.items():
    cols = {'date': DATE_IDX}

    pol, cnt = _align_target(target_daily[ticker])
    cols[f'{ticker}_polarity']  = pol.values
    cols[f'{ticker}_log_count'] = np.log1p(cnt.values)

    for sym in related_syms:
        col = sym.replace('.US', '')
        pol, cnt = _align_related(related_raw.get(sym, []))
        cols[f'{col}_polarity']  = pol.values
        cols[f'{col}_log_count'] = np.log1p(cnt.values)

    out = pd.DataFrame(cols)
    out.to_csv(SENTIMENT_DIR / f'{ticker}.csv', index=False)
    print(f'[{ticker}] {out.shape}  -> sentiment_features/{ticker}.csv')

[AAPL] (2930, 23)  -> sentiment_features/AAPL.csv
[HSBC] (2930, 23)  -> sentiment_features/HSBC.csv
[PEP] (2930, 27)  -> sentiment_features/PEP.csv
[TM] (2930, 27)  -> sentiment_features/TM.csv
[TCEHY] (2930, 25)  -> sentiment_features/TCEHY.csv
